# Day5 - Demo MERGE: Silver + Gold (Bronze → Silver → Gold)

**Objetivo:** Demonstrar o padrão `MERGE INTO` para cargas idempotentes: cria a Silver `pedidos_autoloader` com tipagem correta (`TRY_CAST`), cria a Gold `receita_diaria` por agregação, e depois faz `MERGE` incremental da Bronze para a Silver.

| Item | Valor |
|---|---|
| **Camada** | Silver + Gold |
| **Entrada** | `{catalog}.bronze.pedidos_autoloader` |
| **Saída** | `{catalog}.silver.pedidos_autoloader`, `{catalog}.gold.receita_diaria` |
| **Ordem no pipeline** | Após `DIA4_Gerador_Ingestao_Dados` (Bronze) |

In [0]:
dbutils.widgets.text("catalog", "capgemini_academy", "Nome do catálogo no Unity Catalog")
catalog = dbutils.widgets.get("catalog")

## Silver — Tipagem com TRY_CAST

Cria a Silver `pedidos_autoloader` com `TRY_CAST` (tolera o `pedido_id='null'` que vem sujo da Bronze). `CREATE OR REPLACE TABLE` garante idempotência.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`silver`.`pedidos_autoloader` AS
    SELECT
        TRY_CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        CAST(quantidade AS INT) AS quantidade,
        CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_TIMESTAMP(data_pedido) AS data_pedido
    FROM `${catalog}`.`bronze`.`pedidos_autoloader`
    WHERE 1=1
      AND pedido_id IS NOT NULL
      AND quantidade > 0;

## Gold — Receita diária

Agrega vendas por dia com métricas de negócio (ticket médio, receita total). `CREATE OR REPLACE TABLE` garante idempotência.

In [0]:
%sql
CREATE OR REPLACE TABLE `${catalog}`.`gold`.`receita_diaria` AS
    SELECT
        DATE(data_pedido) AS data_pedido,
        COUNT(DISTINCT pedido_id) AS pedidos,
        SUM(quantidade) AS itens_vendidos,
        SUM(valor_total) AS receita_total,
        ROUND(AVG(valor_total), 2) AS ticket_medio
    FROM `${catalog}`.`silver`.`pedidos_autoloader`
    GROUP BY DATE(data_pedido);

## MERGE INTO — Carga incremental Bronze → Silver

Faz `MERGE` da Bronze (filtrada e tipada) para a Silver existente. Atualiza pedidos que já existem e insere os novos. Idempotente: reexecutar não duplica.

In [0]:
%sql
MERGE INTO `${catalog}`.`silver`.`pedidos_autoloader` AS target
USING (
    SELECT
        TRY_CAST(pedido_id AS BIGINT) AS pedido_id,
        TRIM(cliente_id) AS cliente_id,
        CAST(quantidade AS INT) AS quantidade,
        CAST(valor_total AS DECIMAL(10,2)) AS valor_total,
        TO_TIMESTAMP(data_pedido) AS data_pedido
    FROM `${catalog}`.`bronze`.`pedidos_autoloader`
    WHERE TRY_CAST(pedido_id AS BIGINT) IS NOT NULL
      AND quantidade > 0
) AS source
    ON target.pedido_id = source.pedido_id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;